# HaRP — GPU Run 2: Exp 08, 11, 22b, 22a

Runs 4 GPU experiments on the 700-sample dataset.

## Setup (do this BEFORE running cells):
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Files panel → Upload → `harp_gpu_src.zip` (0.4 MB) — direct upload is fine
3. **For `hidden_states.npz` (94 MB) — use Google Drive (Files panel upload cuts out at ~50 MB):**
   - Go to [drive.google.com](https://drive.google.com) → upload `hidden_states.npz` to **My Drive root**
   - Cell 2 will mount Drive and copy it automatically
4. Run all cells top-to-bottom.

## What runs here:
| Cell | Exp | Description | Time |
|------|-----|-------------|------|
| 4 | 08 | Logit Lens — token prediction across 37 layers (n=100) | ~15 min |
| 5 | 11 | Universal Activation Steering — α sweep (n=100) | ~15 min |
| 6 | 22b | Quadrant-Adaptive Steering, entropy_threshold=0.3745 (data-driven) | ~20 min |
| 7 | 22a | Quadrant-Adaptive Steering, entropy_threshold=0.5 (baseline collapse) | ~20 min |
| **Total** | | | **~70 min** |

## After running:
Last cell downloads `harp_gpu_run2_results.zip`. Extract into your local `harp/`.

In [ ]:
# CELL 1 — Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print('GPU check passed.')

In [ ]:
# CELL 2 — Extract zip, get hidden_states.npz from Drive, validate everything
import zipfile, os, sys, shutil
import numpy as np
from pathlib import Path

src_zip = Path('/content/harp_gpu_src.zip')
if not src_zip.exists():
    raise FileNotFoundError('Upload harp_gpu_src.zip first! (Files panel → Upload)')

# ── Get hidden_states.npz from Google Drive (reliable for large files) ────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Look for hidden_states.npz in My Drive root
drive_path = Path('/content/drive/MyDrive/hidden_states.npz')
if not drive_path.exists():
    raise FileNotFoundError(
        'hidden_states.npz not found in Google Drive root.\n'
        'Go to drive.google.com → upload hidden_states.npz to My Drive (root level), then re-run this cell.'
    )

hs_size_mb = drive_path.stat().st_size / 1024**2
print(f'hidden_states.npz found in Drive: {hs_size_mb:.1f} MB')
if hs_size_mb < 80:
    raise RuntimeError(f'File is only {hs_size_mb:.1f} MB — upload to Drive looks incomplete. Re-upload and try again.')

# ── Extract src zip ───────────────────────────────────────────────────────────
ROOT = Path('/content/harp')
ROOT.mkdir(parents=True, exist_ok=True)

print('Extracting harp_gpu_src.zip ...')
with zipfile.ZipFile(str(src_zip), 'r') as zf:
    zf.extractall(str(ROOT))
print('Done.')

for d in ['results/logs', 'results/plots', 'results/hidden_states']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

# ── Copy hidden_states.npz from Drive ────────────────────────────────────────
hs_dest = ROOT / 'results' / 'hidden_states' / 'hidden_states.npz'
print('Copying hidden_states.npz from Drive (may take ~30s) ...')
shutil.copy(str(drive_path), str(hs_dest))
print(f'Copied → {hs_dest}  ({hs_dest.stat().st_size/1024**2:.1f} MB)')

# ── Validate npz integrity ────────────────────────────────────────────────────
print('Validating hidden_states.npz ...')
try:
    _d    = np.load(str(hs_dest), allow_pickle=True)
    _n    = len(_d['labels'])
    _hall = int((_d['labels'] == 0).sum())
    _corr = int((_d['labels'] == 1).sum())
    _keys = list(_d.keys())
    print(f'  OK — {_n} samples ({_corr} correct, {_hall} hallucinated)')
    print(f'  Keys: {_keys}')
    if _n < 600:
        raise RuntimeError(f'Only {_n} samples found — expected 700. Re-upload hidden_states.npz to Drive.')
except zipfile.BadZipFile as e:
    raise RuntimeError(f'hidden_states.npz is CORRUPTED: {e}. Re-upload to Drive.')

# ── Verify other required files ───────────────────────────────────────────────
os.chdir(str(ROOT))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('\nChecking required files from zip:')
required = [
    'results/hidden_states/steering_vector.npy',
    'results/hidden_states/best_layer_probe.pkl',
    'results/logs/06_probing_results.json',
    'data/processed/features_combined.csv',
]
all_ok = True
for r in required:
    p = ROOT / r
    if p.exists():
        print(f'  OK  {r}  ({p.stat().st_size/1024:.0f} KB)')
    else:
        print(f'  MISSING  {r}')
        all_ok = False

if not all_ok:
    raise RuntimeError('Files missing from harp_gpu_src.zip — rebuild the zip.')

print(f'\nAll checks passed. Working directory: {os.getcwd()}')

In [ ]:
# CELL 3 — Install dependencies
%pip install -q transformers==4.48.0 accelerate scipy scikit-learn tqdm
print('Dependencies installed.')

---
## Experiment 08 — Logit Lens (~15 min)

Projects hidden states at each of the 37 layers through the unembedding matrix.
Shows how token prediction evolves layer-by-layer: correct answers rise in late layers,
hallucinated answers diverge downward. Uses 100 examples (memory constraint).

In [ ]:
# CELL 4 — Exp 08: Logit Lens (~15 min on T4)
import os, runpy
os.chdir('/content/harp')
print('=' * 60)
print('Experiment 08 — Logit Lens')
print('=' * 60)
try:
    runpy.run_path('experiments/08_logit_lens.py', run_name='__main__')
    print('\nExp 08 COMPLETE.')
except SystemExit as e:
    if e.code == 0:
        print('\nExp 08 COMPLETE.')
    else:
        print(f'\nExp 08 FAILED (exit code {e.code}) — continuing to Exp 11.')
except Exception as e:
    import traceback
    print(f'\nExp 08 ERROR: {type(e).__name__}: {e}')
    traceback.print_exc()
    print('Continuing to Exp 11...')

---
## Experiment 11 — Universal Activation Steering (~15 min)

Applies the universal steering vector (computed in Exp 10) at Layer 32 (best layer)
across α ∈ [0.0, 0.5, 1.0, 1.5, 2.0] on 100 hallucinated samples.
Expected finding: α=1.5 optimal (700-sample dataset). Motivates Exp 22b.

In [ ]:
# CELL 5 — Exp 11: Universal Activation Steering (~15 min on T4)
import os, runpy
os.chdir('/content/harp')
print('=' * 60)
print('Experiment 11 — Universal Activation Steering')
print('=' * 60)
try:
    runpy.run_path('experiments/11_run_steering_experiment.py', run_name='__main__')
    print('\nExp 11 COMPLETE.')
except SystemExit as e:
    if e.code == 0:
        print('\nExp 11 COMPLETE.')
    else:
        print(f'\nExp 11 FAILED (exit code {e.code}) — continuing to Exp 22b.')
except Exception as e:
    import traceback
    print(f'\nExp 11 ERROR: {type(e).__name__}: {e}')
    traceback.print_exc()
    print('Continuing to Exp 22b...')

---
## Experiment 22b — Quadrant-Adaptive Steering, entropy_threshold=0.3745 (~20 min)

Data-driven threshold = median entropy of `features_combined.csv` (0.3745). Populates all 4 quadrants:
- **Epistemic** (high entropy, low consistency) — knowledge gap
- **Aleatoric** (high entropy, high consistency) — inherent uncertainty  
- **Confident hallucination** (low entropy, low consistency) — confidently wrong
- **Confident correct** (low entropy, high consistency) — correctly confident

Each quadrant gets its own steering vector. Compare to 22a to see the improvement.

In [ ]:
# CELL 6 — Exp 22b: Quadrant-Adaptive Steering, entropy_threshold=0.3745 (~20 min)
# threshold = median entropy of features_combined.csv (data-driven, populates all 4 quadrants)
import os, shutil, runpy
os.chdir('/content/harp')
os.environ['ENTROPY_THRESHOLD'] = '0.3745'
print('=' * 60)
print('Experiment 22b — Quadrant-Adaptive Steering (entropy=0.3745)')
print('=' * 60)
print(f'ENTROPY_THRESHOLD = {os.environ["ENTROPY_THRESHOLD"]}  (data-driven median)')
try:
    runpy.run_path('experiments/22_quadrant_adaptive_steering.py', run_name='__main__')
    src = '/content/harp/results/logs/22_quadrant_steering_results.json'
    dst = '/content/harp/results/logs/22b_quadrant_steering_results.json'
    shutil.copy(src, dst)
    print(f'\nSaved 22b result → {dst}')
    print('Exp 22b COMPLETE.')
except SystemExit as e:
    if e.code == 0:
        print('\nExp 22b COMPLETE.')
    else:
        print(f'\nExp 22b FAILED (exit code {e.code}) — continuing to Exp 22a.')
except Exception as e:
    import traceback
    print(f'\nExp 22b ERROR: {type(e).__name__}: {e}')
    traceback.print_exc()
    print('Continuing to Exp 22a...')

---
## Experiment 22a — Quadrant-Adaptive Steering, entropy_threshold=0.5 (~20 min)

Baseline: default threshold=0.5. Dataset max entropy is 0.4759 → all samples fall below
0.5 → epistemic and aleatoric quadrants are empty (threshold collapse).
Documents the collapse for comparison against 22b.

In [ ]:
# CELL 7 — Exp 22a: Quadrant-Adaptive Steering, entropy_threshold=0.5 (baseline collapse)
import os, shutil, runpy
os.chdir('/content/harp')
os.environ.pop('ENTROPY_THRESHOLD', None)   # clear 22b's setting → use default 0.5
print('=' * 60)
print('Experiment 22a — Quadrant-Adaptive Steering (entropy=0.5, baseline)')
print('=' * 60)
print('ENTROPY_THRESHOLD = 0.5 (default — max entropy in dataset is 0.4759, so all')
print('samples collapse to low-entropy quadrants — documents threshold failure mode)')
try:
    runpy.run_path('experiments/22_quadrant_adaptive_steering.py', run_name='__main__')
    src = '/content/harp/results/logs/22_quadrant_steering_results.json'
    dst = '/content/harp/results/logs/22a_quadrant_steering_results.json'
    shutil.copy(src, dst)
    print(f'\nSaved 22a result → {dst}')
    print('Exp 22a COMPLETE.')
except SystemExit as e:
    if e.code == 0:
        print('\nExp 22a COMPLETE.')
    else:
        print(f'\nExp 22a FAILED (exit code {e.code}) — packaging whatever results exist.')
except Exception as e:
    import traceback
    print(f'\nExp 22a ERROR: {type(e).__name__}: {e}')
    traceback.print_exc()
    print('Packaging whatever results exist...')

---
## Package & Download Results

Grabs whatever results exist — downloads even if some experiments failed.

In [ ]:
# CELL 8 — Package all available results
import zipfile, os
from pathlib import Path

output_zip = '/content/harp_gpu_run2_results.zip'
root = Path('/content/harp')

# Grab everything that exists in results/logs and results/plots
candidates = [
    'results/logs/08_logit_lens_results.json',
    'results/logs/11_steering_results.json',
    'results/logs/22_quadrant_steering_results.json',
    'results/logs/22a_quadrant_steering_results.json',
    'results/logs/22b_quadrant_steering_results.json',
    'results/plots/08_logit_lens.png',
    'results/plots/11_steering_curve.png',
    'results/plots/22_quadrant_steering_comparison.png',
]

files_added = []
files_missing = []
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pattern in candidates:
        p = root / pattern
        if p.exists():
            arcname = str(p.relative_to(root))
            zf.write(str(p), arcname)
            files_added.append(arcname)
        else:
            files_missing.append(pattern)

size_mb = os.path.getsize(output_zip) / (1024 ** 2)
print(f'Packed {len(files_added)} files ({size_mb:.1f} MB):')
for f in sorted(files_added):
    print(f'  OK  {f}')

if files_missing:
    print(f'\nMissing (experiment may have failed):')
    for f in files_missing:
        print(f'  --  {f}')

if len(files_added) == 0:
    print('\nWARNING: No results found — all experiments failed. Check errors above.')
else:
    print(f'\nReady to download.')

In [ ]:
# CELL 9 — Download results
from google.colab import files
files.download('/content/harp_gpu_run2_results.zip')
print('Download started.')
print()
print('On your local machine, extract with:')
print('  cd ~/Desktop/harp')
print('  unzip ~/Downloads/harp_gpu_run2_results.zip')
print()
print('Then run Exp 23 locally:')
print('  python3 experiments/23_final_evaluation.py')